# Word2Vec — Semantic Analysis of Movie Reviews

This notebook trains a [gensim](https://radimrehurek.com/gensim/) Word2Vec model
on 25,000 movie reviews and explores the learned word embeddings. The reusable
preprocessing and training logic lives in the importable `src/` package, so the
notebook, the training script, and the Streamlit app all share one pipeline.

## 1. Setup

In [1]:
import os

from src.preprocessing import ensure_nltk_data, preprocess
from src.data import load_reviews, tokenize_reviews
from src.train import train_word2vec

# Download the NLTK resources we need, non-interactively (no-op if already present).
ensure_nltk_data()

## 2. Load and preprocess the reviews\n\nWe keep only the `review` text (Word2Vec is unsupervised, so the `sentiment` column is not needed). Preprocessing lower-cases, strips accents and HTML, and removes stop words and very short tokens.

In [2]:
reviews = load_reviews("MovieReview.csv")
print(f"Loaded {len(reviews):,} reviews")
print("Raw example:", reviews[0][:110], "...")

sentences = tokenize_reviews(reviews)
print("Tokenised example:", sentences[0][:12])

avg_len = sum(len(s) for s in sentences) / len(sentences)
print(f"Average tokens per review: {avg_len:.1f}")

Loaded 25,000 reviews
Raw example: With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd doc ...


Tokenised example: ['stuff', 'going', 'moment', 'started', 'listening', 'music', 'watching', 'odd', 'documentary', 'watched', 'wiz', 'watched']
Average tokens per review: 118.0


## 3. Train the Word2Vec model\n\nWe use gensim's CBOW model. Hyper-parameters (vector size, window, epochs, seed) are fixed in `src/train.py` so the model is a reproducible object. `workers=4` speeds training up here; `src.train`'s default `workers=1` is fully deterministic.

In [3]:
model = train_word2vec("MovieReview.csv", workers=4)
wv = model.wv
print(f"Vocabulary size: {len(wv):,}")
print(f"Vector dimension: {wv.vector_size}")

Vocabulary size: 28,338
Vector dimension: 100


## 4. Explore the embeddings\n\n### Nearest neighbours\n\nThe most semantically similar words by cosine similarity.

In [4]:
for word in ["movie", "good", "terrible", "actor"]:
    neighbours = [w for w, _ in wv.most_similar(word, topn=6)]
    print(f"{word:10s} -> {', '.join(neighbours)}")

movie      -> film, movies, flick, thats, honestly, think
good       -> decent, great, alright, bad, okay, fine
terrible   -> horrible, awful, atrocious, dreadful, horrendous, abysmal
actor      -> actress, actors, role, comedian, pacino, roles


### Word analogies\n\nVector arithmetic such as *king − man + woman*. Note that reliable analogy solving needs a large, general corpus; on 25k movie reviews the results are indicative rather than perfect (issue #12 evaluates this quantitatively).

In [5]:
result = wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
for w, score in result:
    print(f"{w:12s} {score:.3f}")

furst        0.480
queen        0.477
lion         0.458


## 5. Interactive demo (`app.py`)

The trained vectors power a Streamlit app with the same two features (similarity
and analogies). Save the model and launch the app from the repo root:

```python
model.save("word2vec.model")
model.wv.save("word2vec.wv")
```

```bash
streamlit run app.py
```

`app.py` normalises user input through the same `src/` preprocessing used here,
so lookups always match the trained vocabulary.